<a href="https://colab.research.google.com/github/meredithcfarley/cs-74-crisis-detection/blob/main/03analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# cell 1
# load everything from drive from weeks 1-3 (from preprocessing and models notebooks) to do analysis notebook here


import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/CS74"
RESULTS_DIR = f"{BASE}/cv_results"

#data and the same split as before!
df = pd.read_parquet(f"{BASE}/cleaned.parquet")
X = df[["clean_text", "word_count"]]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

#load everything saved from 02 models
nb_final = joblib.load(f"{RESULTS_DIR}/nb_final_model.joblib")
lr_final = joblib.load(f"{RESULTS_DIR}/lr_final_model.joblib")
chosen_thresholds = joblib.load(f"{RESULTS_DIR}/thresholds.joblib")
test_probs = joblib.load(f"{RESULTS_DIR}/test_probs.joblib")
nb_test_probs = test_probs["nb"]
lr_test_probs = test_probs["lr"]

print("Test set:", X_test.shape)
print("Thresholds:", chosen_thresholds)

Mounted at /content/drive
Test set: (46403, 2)
Thresholds: {'naive_bayes': np.float64(0.4419177149427485), 'logistic_regression': np.float64(0.3690998126388735)}


In [2]:
#cell 2
# list the terms with the biggest weights to see what the model relies on

tfidf = lr_final.named_steps["tfidf"] #numbers
clf = lr_final.named_steps["clf"] #classifier that decides class

#list all kept 10k terms
terms = tfidf.get_feature_names_out()
#list of all LRs 10k weights in the same order - so they correspond!
weights = clf.coef_[0]

#puts the weights and terms side by side in a table
weight_table = pd.DataFrame({"term": terms, "weight": weights})

#sorts from biggest to smallest and keeps the top 20 strongest crisis terms
top_crisis = weight_table.sort_values("weight", ascending=False).head(20).reset_index(drop=True)
#sorts from smallest up (negative, bringing to 0) to find the strongest non crisis terms
top_noncrisis = weight_table.sort_values("weight").head(20).reset_index(drop=True)

side_by_side = pd.concat([top_crisis.add_prefix("crisis_"), top_noncrisis.add_prefix("noncrisis_")], axis=1).round(2)

side_by_side.to_csv(f"{BASE}/lr_top_terms.csv", index=False)
side_by_side

,crisis_term,crisis_weight,noncrisis_term,noncrisis_weight
0,suicidal,23.31,teenagers,-14.66
1,suicide,23.18,rant,-10.95
2,pills,10.64,horny,-10.70
3,kill yourself,10.63,minecraft,-10.03
4,overdose,9.80,memes,-9.20
5,hopeless,9.65,crush,-9.09
6,kill myself,9.31,bruh,-9.06
7,hang myself,9.17,teenager,-8.70
8,life,8.99,filler,-8.55
9,medication,8.93,cute,-8.54


The crisis words are explicit/direct, the non crisis words are teenager based (the model is using shortcuts)

In [3]:
#cell 3
#slang vocabulary check
#what slang/indirect phrases are in the 10k term vocabulary
#if people use not direct terms, what else do they use?

#list of words to check for
check = ["unalive", "kms", "kys", "ctb", "sewerslide", "wanna die", "no point", "give up", "so tired", "not worth", "am done"]

vocab = tfidf.vocabulary_

rows = []
for t in check:
  #check if the term made the 10k cut
    if t in vocab:
        rows.append({"term": t, "in vocab": "yes", "weight": round(weights[vocab[t]], 2)})
    else:
        rows.append({"term": t, "in vocab": "no", "weight": None})

slang_table = pd.DataFrame(rows)
slang_table.to_csv(f"{BASE}/slang_check.csv", index=False)
slang_table

,term,in vocab,weight
0,unalive,no,NaN
1,kms,yes,4.91
2,kys,no,NaN
3,ctb,no,NaN
4,sewerslide,no,NaN
5,wanna die,no,NaN
6,no point,yes,2.16
7,give up,yes,3.68
8,so tired,yes,0.21
9,not worth,yes,2.71


In [4]:
#cell 4
# is teenagers as the top non crisis word not just leakage from r/teenagers?

text = X_test["clean_text"]
#marks post true if it contains a teenager word
mentions_teen = text.str.contains(r"\b(?:teenager|teenagers|teens)\b")

#build a table of how many crisis vs non crisis posts mention the word
print("Test posts mentioning teenager(s)/teens:", mentions_teen.sum())
print(pd.crosstab(mentions_teen, y_test, rownames=["mentions teen word"], colnames=["label (1 = crisis)"]))

Test posts mentioning teenager(s)/teens: 900
label (1 = crisis)      0      1
mentions teen word              
False               22597  22906
True                  600    300


Is the model just relying on the word teenager to classify non crisis posts? its a 2:1 ratio for if teenager is present, is it a non crisis or a crisis post. The model leans on it heavily to classify non crisis posts. However, the real concern is the mention of teenager in crisis posts. in this case, the presence of the word teenager pushes the model away from flagging it, since it is negatively weighted to the non crisis class. this is under flagging at risk teens

In [5]:
lr_flag = lr_test_probs >= chosen_thresholds["logistic_regression"]
crisis = (y_test == 1).values
m = mentions_teen.values

print("LR recall, crisis posts mentioning teen words:", lr_flag[crisis & m].mean().round(3))
print("LR recall, all other crisis posts:            ", lr_flag[crisis & ~m].mean().round(3))

LR recall, crisis posts mentioning teen words: 0.977
LR recall, all other crisis posts:             0.952


Well I guess I was wrong! Crisis recall is actually higher when teenager words are present. The presence of teenager words isn't pulling the model in the wrong direction like i hypothesized. Its possible that other words in the post outweigh the teenager mention, or also that these posts are longer and more detailed and therefore flagged.

In [6]:
#cell 5
# this cell creates 100 sample posts to be used for high confidence error auditing and hyperbole analysis

import re

#build one table with everything needed for the audit
d = X_test.copy()
d["text"] = df.loc[X_test.index, "text"]
d["label"] = y_test.values
d["nb_prob"] = nb_test_probs
d["lr_prob"] = lr_test_probs
d["post_id"] = d.index

N = 25

#high confidence errors in LR: crisis posts below 0.1, non crisis posts above 0.9
fn_all = d[(d.label == 1) & (d.lr_prob < 0.1)]
fp_all = d[(d.label == 0) & (d.lr_prob > 0.9)]
missed = fn_all.sample(N, random_state=42)
false_alarms = fp_all.sample(N, random_state=42)

#hyperbole candidates: 25 from each class containing a hyperbole phrase
pat = re.compile(r"\b(?:kill me|kms|want to die|wanna die|end me|kill myself)\b", re.I)
cand = d[d["text"].str.contains(pat)]
hyperbole = pd.concat([cand[cand.label == 0].sample(N, random_state=42),cand[cand.label == 1].sample(N, random_state=42),])

print("High-confidence misses:", len(fn_all), "| high-confidence false alarms:", len(fp_all))
print("Missed crisis post_ids:", sorted(missed.post_id.tolist()))

High-confidence misses: 348 | high-confidence false alarms: 338
Missed crisis post_ids: [7021, 7072, 8594, 26586, 30751, 36823, 37930, 52430, 60288, 65532, 80340, 103892, 108640, 123531, 134240, 137176, 154108, 156564, 173005, 176751, 178693, 179246, 184168, 196832, 209137]
